# EDA: Biased Supply Air Temperature Sensor (Experimental Dataset)

## Fault details (per LBNL documentation)

Four severities: bias of +2°C, +4°C, -2°C, -4°C applied to the SAT sensor reading.
**Critical documented detail**: "the supply air temperature in the dataset is the
faulty value" — meaning `RTU_SA_TEMP` in these files is already the biased reading,
not a separate true-vs-sensed pair. This is fundamentally different from the other
two fault types: it's a sensor fault, not a physical/mechanical one, and the dataset
gives us only the corrupted signal, not ground truth to compare it against directly.

## Why this fault is a different kind of detection problem

OA damper stuck and incorrect economizer setpoint both affect physical airflow/control
behavior, visible on multiple correlated signals. A biased SAT sensor, by contrast, is
purely a reporting error — the physical system may be operating completely normally,
but what it *reports* about SAT is wrong. This means the detection strategy has to be
fundamentally different: not "does this signal show an unexpected physical state" but
"is this signal internally inconsistent with what other signals imply it should be."

## Hypothesis (before looking at any data)

- Since `RTU_SA_TEMP` is the setpoint the compressor/furnace sequencing controls
  *to* (55°F target, per the documented control sequence), a biased sensor reading
  should cause the control system to actually chase the wrong target — meaning the
  fault should be detectable not on SAT itself (which will report ~55°F regardless,
  since that's what the control loop is chasing) but on a **derived inconsistency**:
  the real physical effects of chasing the wrong setpoint (e.g. compressor run time
  changes, zone temperature drift) should differ from baseline, even though SAT
  itself won't visibly show the bias.
- This is a genuinely different, and arguably harder, detection challenge than
  anything in this dataset so far — worth checking directly whether zone temperature
  (`ZA_TEMP`) or compressor power (`RTU_COMP_WATT_1`/`_2`) shows a real, checkable
  deviation, since those would be the practical signals a real FDD system would need.
- Given the recurring pattern in this dataset of raw means being misleading without
  checking the underlying mechanism (staging, occupancy, weather confounds), plan to
  check the actual time series/scatter before trusting any single summary number.

In [1]:
import pandas as pd

files = {
    "baseline": "../data/raw/experimental/ERTU_Winter_2022.csv",
    "sat_bias_neg4": "../data/raw/experimental/SA_temp_bias_-4_Winter_2022.csv",
    "sat_bias_neg2": "../data/raw/experimental/SA_temp_bias_-2_Winter_2022.csv",
    "sat_bias_pos2": "../data/raw/experimental/SA_temp_bias_2_Winter_2022.csv",
    "sat_bias_pos4": "../data/raw/experimental/SA_temp_bias_4_Winter_2022.csv",
}

dfs_sat = {label: pd.read_csv(fname, na_values=["NAN"]) for label, fname in files.items()}

for _label, df in dfs_sat.items():
    df["Datetime"] = pd.to_datetime(df["Datetime"])

for _label, df in dfs_sat.items():
    print(f"{_label}: shape={df.shape}, missing_values={df.isna().sum().sum()}")

baseline: shape=(2880, 57), missing_values=12155
sat_bias_neg4: shape=(1440, 57), missing_values=5187
sat_bias_neg2: shape=(1440, 57), missing_values=949
sat_bias_pos2: shape=(1440, 57), missing_values=6747
sat_bias_pos4: shape=(1440, 57), missing_values=5811


## Load confirmed

All five files load with real, non-zero missing-value counts (consistent with the
documented unoccupied-mode sensor dropout). `sat_bias_neg2`'s much lower missing count
(949 vs. others in the 5,000-6,800 range) is worth a quick note, plausibly just a
different occupied/unoccupied split for that specific day, same reasoning as OA damper
stuck's varying missing counts — not chasing further unless it distorts a real
comparison.

In [2]:
print([c for c in dfs_sat["baseline"].columns if "TEMP" in c or "TERM" in c])

['RTU_MA_TEMP', 'RTU_OA_TEMP', 'RTU_RA_TEMP', 'RTU_SA_TEMP', 'TERM_RM_HUMD_102', 'TERM_RM_HUMD_103', 'TERM_RM_HUMD_104', 'TERM_RM_HUMD_105', 'TERM_RM_HUMD_106', 'TERM_RM_HUMD_202', 'TERM_RM_HUMD_203', 'TERM_RM_HUMD_204', 'TERM_RM_HUMD_205', 'TERM_RM_HUMD_206', 'TERM_RM_TEMP_102', 'TERM_RM_TEMP_103', 'TERM_RM_TEMP_104', 'TERM_RM_TEMP_105', 'TERM_RM_TEMP_106', 'TERM_RM_TEMP_202', 'TERM_RM_TEMP_203', 'TERM_RM_TEMP_204', 'TERM_RM_TEMP_205', 'TERM_RM_TEMP_206', 'TERM_RM_TEMP_CSPT', 'TERM_RM_TEMP_HSPT']


In [3]:
sat_cols = ["RTU_SA_TEMP", "RTU_RA_TEMP", "RTU_COMP_WATT_1", "RTU_COMP_WATT_2", "RTU_TOT_WATT"]
severity_order_sat = ["baseline", "sat_bias_neg4", "sat_bias_neg2", "sat_bias_pos2", "sat_bias_pos4"]

summary_sat = pd.DataFrame({
    label: dfs_sat[label][dfs_sat[label]["OCCU_MOD"] == 1][sat_cols].mean()
    for label in severity_order_sat
}).T.loc[severity_order_sat]

summary_sat

,RTU_SA_TEMP,RTU_RA_TEMP,RTU_COMP_WATT_1,RTU_COMP_WATT_2,RTU_TOT_WATT
baseline,56.327350,67.549397,1515.008513,2.699406,3407.935516
sat_bias_neg4,56.455828,72.248533,2163.935544,187.868808,4759.791349
sat_bias_neg2,56.534250,70.309978,2367.745313,350.802731,5193.905681
sat_bias_pos2,56.007322,66.947022,2269.243151,480.817449,4969.385047
sat_bias_pos4,56.020372,66.626033,3588.135357,748.508017,6911.142205


## Finding: SAT bias is invisible on SAT itself, exactly as hypothesized — but produces
## a large, clean, monotonic signature on compressor 2 activation

| Severity | SA_TEMP | RA_TEMP | COMP_WATT_1 | COMP_WATT_2 | TOT_WATT |
|---|---|---|---|---|---|
| baseline | 56.33°F | 67.55°F | 1515.0W | **2.7W** | 3407.9W |
| bias -4°C | 56.46°F | 72.25°F | 2163.9W | 187.9W | 4759.8W |
| bias -2°C | 56.53°F | 70.31°F | 2367.7W | 350.8W | 5193.9W |
| bias +2°C | 56.01°F | 66.95°F | 2269.2W | 480.8W | 4969.4W |
| bias +4°C | 56.02°F | 66.63°F | 3588.1W | **748.5W** | 6911.1W |

**`RTU_SA_TEMP` confirms the hypothesis exactly**: essentially flat (56.0-56.5°F)
regardless of bias direction or magnitude — the control loop chases the biased
reading, making SAT itself blind to this fault by construction, matching the
documentation's note that the dataset's SAT column already contains the faulty value.

**`RTU_COMP_WATT_2` is the standout signal**: baseline shows compressor 2 essentially
never engaging in winter (2.7W, effectively off) — every biased scenario forces it on,
scaling with bias magnitude in both directions (187.9W at -4°C up to 748.5W at +4°C,
roughly 70-280x baseline). This is a real, large, and directionally interpretable
effect: a biased sensor makes the system chase a wrong target hard enough to force
extra compressor staging that wouldn't otherwise be needed.

**`RTU_TOT_WATT` shows a large, easily-measurable overall effect** — up to ~2x
baseline at the most severe setting — a strong, simple energy-cost signal that could
directly support the "estimated energy impact" framing discussed earlier for the
eventual copilot output.

**Practical implication for modeling**: this fault is the clearest example so far of
"invisible on the sensor most people would check, obvious on a derived/adjacent
signal" — exactly the kind of finding that justifies real FDD tooling over a simple
threshold alert on SAT alone. `RTU_COMP_WATT_2` and `RTU_TOT_WATT` are strong candidate
features; `RTU_SA_TEMP` should be explicitly excluded from this fault's feature set
despite being the fault's namesake sensor, since it structurally cannot reveal it.

## Generalization check: does the same pattern hold in another season?

Checking Spring_2021, since it's the next available season and (per notebook 07)
also showed real economizer activity in its baseline, making it a reasonable
comparison point — though the relevant mechanism here (compressor staging under a
biased SAT target) is less directly tied to the economizer than the previous two
fault types were, so this is more of a sanity check than a targeted choice.

In [4]:
files_spring_sat = {
    "baseline": "../data/raw/experimental/ERTU_Spring_2021.csv",
    "sat_bias_neg4": "../data/raw/experimental/SA_temp_bias_-4_Spring_2021.csv",
    "sat_bias_neg2": "../data/raw/experimental/SA_temp_bias_-2_Spring_2021.csv",
    "sat_bias_pos2": "../data/raw/experimental/SA_temp_bias_2_Spring_2021.csv",
    "sat_bias_pos4": "../data/raw/experimental/SA_temp_bias_4_Spring_2021.csv",
}

dfs_spring_sat = {label: pd.read_csv(fname, na_values=["NAN"]) for label, fname in files_spring_sat.items()}
for _label, df in dfs_spring_sat.items():
    df["Datetime"] = pd.to_datetime(df["Datetime"])

summary_spring_sat = pd.DataFrame({
    label: dfs_spring_sat[label][dfs_spring_sat[label]["OCCU_MOD"] == 1][sat_cols].mean()
    for label in severity_order_sat
}).T.loc[severity_order_sat]

summary_spring_sat

,RTU_SA_TEMP,RTU_RA_TEMP,RTU_COMP_WATT_1,RTU_COMP_WATT_2,RTU_TOT_WATT
baseline,56.091869,68.370506,1462.137704,85.911594,3632.502371
sat_bias_neg4,56.085022,72.245739,1878.515683,2.837916,4236.899823
sat_bias_neg2,56.120200,72.053478,3128.223683,262.457980,6115.363621
sat_bias_pos2,56.638856,69.892744,3687.345455,268.675538,6811.931519
sat_bias_pos4,50.481700,68.682983,4443.877835,567.192273,7881.259094


## Generalization check: two findings partially replicate, one genuinely doesn't —
## season matters here more than for the other two fault types

| Severity | SA_TEMP (Winter) | SA_TEMP (Spring) | TOT_WATT (Winter) | TOT_WATT (Spring) |
|---|---|---|---|---|
| baseline | 56.33°F | 56.09°F | 3407.9W | 3632.5W |
| bias -4°C | 56.46°F | 56.09°F | 4759.8W | 4236.9W |
| bias -2°C | 56.53°F | 56.12°F | 5193.9W | 6115.4W |
| bias +2°C | 56.01°F | 56.64°F | 4969.4W | 6811.9W |
| bias +4°C | 56.02°F | **50.48°F** | 6911.1W | 7881.3W |

**`RTU_TOT_WATT` monotonic-increase pattern replicates directionally** in both
seasons (roughly 1.4-2.2x baseline at the most severe settings) — this is a robust
finding across two seasons.

**`RTU_SA_TEMP`'s "always invisible" claim does NOT fully replicate** — Spring's
sat_bias_pos4 shows a real 5.6°F drop, breaking the compensation pattern seen at every
other severity/season combination checked so far. This is a genuine, meaningful
exception, not noise (5.6°F is a large deviation compared to the ~0.5°F range seen
everywhere else). Not yet explained — possibly the compensation mechanism has a limit
that gets exceeded at the most extreme bias combined with Spring's different baseline
conditions, but this is speculation, not confirmed.

**`RTU_COMP_WATT_2`'s clean story from Winter doesn't hold in Spring** — baseline
compressor-2 activity is already substantial in Spring (85.9W vs. Winter's 2.7W),
making the "fault forces otherwise-idle compressor 2 on" narrative specific to Winter
conditions, not a universal mechanism.

**Revised, more honest conclusion**: SAT bias's effect on `RTU_TOT_WATT` (real,
directional, energy-cost-relevant) is the most robust cross-season finding for this
fault. The specific mechanism (SAT staying invisible, compressor-2 forced on) is
real in Winter but doesn't fully generalize — season-dependent behavior, consistent
with the pattern already seen in this Experimental dataset (OA damper stuck's
Winter-vs-Spring damper_005 anomaly, economizer setpoint's season-dependent valid
test windows). This dataset seems to consistently require checking generalization
before trusting a single-season finding, more so than the Simulated dataset did.

## Summary: biased SAT sensor EDA

**Confirmed, robust across two seasons**: `RTU_TOT_WATT` increases meaningfully and
directionally with bias magnitude — a genuine, energy-cost-relevant signal.

**Confirmed in Winter, partially broken in Spring**: `RTU_SA_TEMP` mostly stays
invisible to this fault (control loop chases the biased value), except Spring's most
extreme severity (+4°C), which showed a real 5.6°F deviation — an honestly-flagged
exception, not smoothed over.

**Not a robust, cross-season signal**: `RTU_COMP_WATT_2`'s clean "forces an otherwise-
idle compressor on" story was specific to Winter's baseline conditions and doesn't
generalize to Spring, where compressor 2 is already active at baseline.

**Practical implication for modeling**: `RTU_TOT_WATT` is the most defensible
cross-season feature for this fault. `RTU_SA_TEMP` should not be assumed universally
blind to this fault — worth including it as a feature anyway, since Spring's +4°C
exception suggests it may carry real signal at extreme severities even if not at
moderate ones.